## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import os

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (load_and_prepare_data, harmo_labels, analysis_groups_clusters,
                            harmonisation_order, analysis_conditions, color_strips_by_condition,
                            label_strips_by_condition, surf_feature_names,
                            friedman_across_conditions, describe_across_conditions, NVERT,
                            load_surf_feature_maps)


In [ ]:
eval_stats_df = load_and_prepare_data()

## read the surface features per vertex

In [ ]:
surf_feature_maps, cortex_mask, feature_rows = load_surf_feature_maps(eval_stats_df)

## compare mean features in GT labels

In [ ]:
# mean of each surface feature inside the ground-truth lesion label, per patient and
# condition, taken from the vertexwise maps above. the label is drawn once and projected to
# fsaverage_sym from the 3T reconstruction by vol_eval.py, so it is the same lesion in every
# condition; it is usually on one hemisphere only and absent for controls
gt_masks = {}
for _, subject in feature_rows.drop_duplicates(subset='site_subj_id').iterrows():
    masks = []
    for hemi in ['lh', 'rh']:
        gt_path = (f'data/results/labeled_predictions/gt_fsaverage_sym/{subject["source"]}'
                   f'/{subject["subject ID"]}/{hemi}.gt.fsaverage_sym.mgh')
        masks.append((nib.load(gt_path).get_fdata().squeeze() == 1) & cortex_mask
                     if os.path.exists(gt_path) else np.zeros(NVERT, dtype=bool))

    if any(mask.any() for mask in masks):
        gt_masks[subject['site_subj_id']] = np.stack(masks)

gt_records = []
for (harmo, analysis_group, site_subj_id, feature), maps in surf_feature_maps.items():
    if site_subj_id not in gt_masks:
        continue

    # both hemispheres pooled, so that a label crossing the midline gives one mean and not two
    values = maps[gt_masks[site_subj_id]]
    values = values[np.isfinite(values)]
    if values.size == 0:
        print(f'{site_subj_id} has no {feature} under its ground-truth label in {analysis_group} '
              f'{harmo}, skipped.')
        continue

    gt_records.append({'harmo': harmo, 'analysis_group': analysis_group,
                       'site_subj_id': site_subj_id, 'feature': feature,
                       'n_vertices': values.size,
                       'mean_value_norm': values.mean(), 'sd_value_norm': values.std()})

# merging onto the rows the maps were extracted from picks up the site, the group and whether
# meld_graph found the lesion in that condition
gt_features_df = pd.merge(
    pd.DataFrame(gt_records),
    feature_rows[['harmo', 'analysis_group', 'site_subj_id',  # matching on these
                  'B0_condition', 'site', 'group', 'tp_patient']],  # these added
    on=['harmo', 'analysis_group', 'site_subj_id'],
    how='inner',
    validate='many_to_one')

# the same six-condition label as for the clusters
gt_features_df['harmonisation'] = gt_features_df['harmo'].map(harmo_labels)
gt_features_df['analysis_condition'] = (gt_features_df['analysis_group'] + ' ' +
                                        gt_features_df['harmonisation'])

gt_features_df


In [ ]:
# the ground-truth labels are the same lesion in every condition, so these stay a paired test: the
# Friedman test per feature instead of per metric, run once per set of conditions - across all six,
# and across the three acquisition conditions of each harmonisation on its own. the p-values are
# corrected across the features within each such run
friedman_conditions_all = 'all six conditions'
friedman_condition_sets = {friedman_conditions_all: analysis_conditions}
friedman_condition_sets.update({
    f'{harmonisation} only': [f'{analysis_group} {harmonisation}'
                              for analysis_group in analysis_groups_clusters]
    for harmonisation in harmonisation_order})

gt_features_paired = {}
p_values_gt = []

for conditions_label, conditions in friedman_condition_sets.items():
    paired, p_values_df = friedman_across_conditions(
        gt_features_df, conditions, key_col='feature', value_col='mean_value_norm')
    if len(paired) == 0:
        print(f'No complete cases for {conditions_label}, skipped.')
        continue

    # describe exactly the complete cases the Friedman test ran on. the subsets share their
    # column names with the six-condition run, so the rows line up in one table
    p_values_df = pd.merge(
        describe_across_conditions(paired.melt(id_vars=['site_subj_id', 'feature'],
                                               var_name='analysis_condition',
                                               value_name='mean_value_norm'),
                                   conditions,
                                   key_col='feature', value_col='mean_value_norm'),
        p_values_df, on='feature')

    gt_features_paired[conditions_label] = paired
    p_values_gt.append(p_values_df.assign(conditions=conditions_label))

p_values_gt_df = pd.concat(p_values_gt, ignore_index=True)
# label column first
p_values_gt_df = p_values_gt_df[['conditions'] +
                                [col for col in p_values_gt_df.columns if col != 'conditions']]

paired_all_conditions = gt_features_paired[friedman_conditions_all]
print(f"{paired_all_conditions['site_subj_id'].nunique()} patients with ground-truth label "
      f"features in every condition")
paired_all_conditions


In [ ]:
p_values_gt_df

### Supplementary Figure 5

In [ ]:
# swarm plots of the normalised per-patient mean inside the ground-truth lesion label, one column
# per feature and the six conditions on each x-axis. the marker says whether meld_graph found that
# patient's lesion in that condition, so that the values of the detected and the missed lesions can
# be told apart within each condition
gt_features_plot = gt_features_df.copy()
gt_features_plot['feature'] = gt_features_plot['feature'].map(surf_feature_names)

# the panels of this figure are ordered by feature, not in the order surf_feature_names lists
# them; both the columns and the per-point markers below follow this order
gt_feature_order = ['Cortical thickness', 'White-grey contrast', 'Intrinsic curvature',
                    'Mean curvature', 'Sulcal depth']
assert set(gt_feature_order) == set(surf_feature_names.values())

grid = sns.catplot(
    data=gt_features_plot,
    x='analysis_group',
    order=analysis_groups_clusters,
    y='mean_value_norm',
    hue='harmonisation',
    hue_order=harmonisation_order,
    palette=['lightgray'] * len(harmonisation_order),
    legend=False,
    col='feature',
    col_order=gt_feature_order,
    col_wrap=3,
    kind='strip',
    dodge=True,
    jitter=0.15,
    size=4,
    sharey=False,
    sharex=False,
    height=3.0,
    aspect=1.1,
)
color_strips_by_condition(grid)

# seaborn gives every collection it draws a single marker path, but a collection can hold one
# path per point instead - which is what makes a per-patient marker possible. each point is
# matched back to its row by its value, which the strip plot leaves untouched (only x is
# jittered), so this does not depend on how seaborn ordered the points
marker_paths = {}
for detected, marker in [(True, 'X'), (False, 'o')]:
    marker_style = matplotlib.markers.MarkerStyle(marker)
    marker_paths[detected] = marker_style.get_path().transformed(marker_style.get_transform())

for ax, feature_label in zip(grid.axes.flat, gt_feature_order):
    points = gt_features_plot[gt_features_plot['feature'] == feature_label]
    detected_by_value = dict(zip(points['mean_value_norm'], points['tp_patient']))
    for collection in ax.collections:
        collection.set_paths([marker_paths[detected_by_value.get(value) == True]
                              for value in collection.get_offsets()[:, 1]])

grid.fig.legend(handles=[matplotlib.lines.Line2D([], [], linestyle='', marker=marker,
                                                 markersize=4,
                                                 color='dimgrey', label=label)
                         for marker, label in [('x', 'lesion detected'), ('o', 'lesion missed')]],
                frameon=False,
                bbox_to_anchor=(0.70, 0.4),
                loc='center left')

grid.set_axis_labels('', 'mean in FCD lesion label\n[z-score]')
grid.set_titles('{col_name}')
label_strips_by_condition(grid)

# add a horizontal line at 0, the mean of the normalised feature across the whole cortex
for ax in grid.axes:
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.5)

# add vertical spacing between the rows of panels
grid.fig.subplots_adjust(hspace=0.9)


letters = ['A', 'B', 'C', 'D', 'E']
for ax in grid.axes.flat:
    plt.text(0.02, 0.98, letters.pop(0), transform=ax.transAxes, fontsize=11, va='top', ha='left')